In [1]:
! pwd

/workspace


In [2]:
! ls /workspace/data/retail_db/datasets/orders

orders.csv


In [3]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.\
        builder.\
        appName("pyspark-notebook").\
        master("spark://spark-master:7077").\
        config("spark.executor.memory", "512m").\
        getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/03 12:48:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
orders_rdd = spark.sparkContext.textFile("/workspace/data/retail_db/datasets/orders/*")

In [6]:
orders_rdd.take(5)

['1,2013-07-25 00:00:00.0,11599,CLOSED',
 '2,2013-07-25 00:00:00.0,256,PENDING_PAYMENT',
 '3,2013-07-25 00:00:00.0,12111,COMPLETE',
 '4,2013-07-25 00:00:00.0,8827,CLOSED',
 '5,2013-07-25 00:00:00.0,11318,COMPLETE']

In [7]:
spark

## 1. Count the orders under each status

In [13]:
mapped_rdd = orders_rdd.map(lambda x: (x.split(",")[3],1))

In [14]:
mapped_rdd.take(10)

[('CLOSED', 1),
 ('PENDING_PAYMENT', 1),
 ('COMPLETE', 1),
 ('CLOSED', 1),
 ('COMPLETE', 1),
 ('COMPLETE', 1),
 ('COMPLETE', 1),
 ('PROCESSING', 1),
 ('PENDING_PAYMENT', 1),
 ('PENDING_PAYMENT', 1)]

In [15]:
reduced_rdd = mapped_rdd.reduceByKey(lambda x,y :x+y)

In [16]:
reduced_rdd.collect()

[('CLOSED', 7556),
 ('CANCELED', 1428),
 ('PENDING_PAYMENT', 15030),
 ('COMPLETE', 22899),
 ('PROCESSING', 8275),
 ('PAYMENT_REVIEW', 729),
 ('PENDING', 7610),
 ('ON_HOLD', 3798),
 ('SUSPECTED_FRAUD', 1558)]

In [19]:
# sorted_rdd = reduced_rdd.sortBy(lambda x:x[1])   # By default Ascending is true

sorted_rdd = reduced_rdd.sortBy(lambda x:x[1], False)          # False means Ascending false, do it as Descending order.

In [20]:
sorted_rdd.collect()

[('COMPLETE', 22899),
 ('PENDING_PAYMENT', 15030),
 ('PROCESSING', 8275),
 ('PENDING', 7610),
 ('CLOSED', 7556),
 ('ON_HOLD', 3798),
 ('SUSPECTED_FRAUD', 1558),
 ('CANCELED', 1428),
 ('PAYMENT_REVIEW', 729)]

## 2. Find the premium customers ( top 10 who placed the most no. of orders)

In [23]:
customers_mapped = orders_rdd.map(lambda x: (x.split(",")[2],1))

In [24]:
customers_mapped.take(5)

[('11599', 1), ('256', 1), ('12111', 1), ('8827', 1), ('11318', 1)]

In [25]:
customers_agg = customers_mapped.reduceByKey(lambda x,y : x+y)

In [26]:
customers_agg.take(10)

[('256', 10),
 ('12111', 6),
 ('11318', 6),
 ('7130', 7),
 ('2911', 6),
 ('5657', 12),
 ('9149', 4),
 ('9842', 7),
 ('7276', 5),
 ('9488', 7)]

In [29]:
customers_sorted = customers_agg.sortBy(lambda x: x[1], False)

In [30]:
customers_sorted.take(10)

[('569', 16),
 ('6316', 16),
 ('12431', 16),
 ('5897', 16),
 ('12284', 15),
 ('5654', 15),
 ('5283', 15),
 ('4320', 15),
 ('221', 15),
 ('5624', 15)]

## 3. Distinct count of customers who placed atleast one order

In [31]:
dist_mapp = orders_rdd.map(lambda x: x.split(",")[2]).distinct()

### Here count is an action

In [32]:
dist_mapp.count()

12405

In [33]:
## Total orders

orders_rdd.count()

68883

## 4. Which customer has the most number of CLOSED order

In [37]:
filtered_closed_mapp = orders_rdd.filter(lambda x : x.split(",")[3] == 'CLOSED')

In [38]:
filtered_closed_mapp.take(10)

['1,2013-07-25 00:00:00.0,11599,CLOSED',
 '4,2013-07-25 00:00:00.0,8827,CLOSED',
 '12,2013-07-25 00:00:00.0,1837,CLOSED',
 '18,2013-07-25 00:00:00.0,1205,CLOSED',
 '24,2013-07-25 00:00:00.0,11441,CLOSED',
 '25,2013-07-25 00:00:00.0,9503,CLOSED',
 '37,2013-07-25 00:00:00.0,5863,CLOSED',
 '51,2013-07-25 00:00:00.0,12271,CLOSED',
 '57,2013-07-25 00:00:00.0,7073,CLOSED',
 '61,2013-07-25 00:00:00.0,4791,CLOSED']

In [ ]:
## Paired rdd is nothing but (x,y) inside an colsed paranthesis, two entities inside,

In [39]:
filter_mapped = filtered_closed_mapp.map(lambda x: (x.split(",")[2] , 1))

In [40]:
filter_mapped.take(10)

[('11599', 1),
 ('8827', 1),
 ('1837', 1),
 ('1205', 1),
 ('11441', 1),
 ('9503', 1),
 ('5863', 1),
 ('12271', 1),
 ('7073', 1),
 ('4791', 1)]

In [41]:
filter_agg = filter_mapped.reduceByKey(lambda x,y : x+y)

In [42]:
filter_agg.take(10)

[('3159', 1),
 ('5834', 2),
 ('10173', 1),
 ('2101', 1),
 ('6000', 1),
 ('1352', 2),
 ('10142', 1),
 ('12210', 1),
 ('6018', 2),
 ('2252', 1)]

In [44]:
sorted_filter = filter_agg.sortBy(lambda x: x[1], False)

In [45]:
sorted_filter.take(10)

[('1833', 6),
 ('1687', 5),
 ('1363', 5),
 ('5493', 5),
 ('7879', 4),
 ('7948', 4),
 ('4282', 4),
 ('4573', 4),
 ('8974', 4),
 ('2321', 4)]

---

In [46]:
spark.stop()